# 0326 LangGraph Smoketest

Smoke test for `langgraph_agent_runner.py` on a small subset of SPL IDs (`n=5`).

This notebook intentionally stops after `aggregated_rows` and does **not** run evaluation.

In [2]:
import os,torch
os.environ["RUNNER_CUDA_VISIBLE_DEVICES"] = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
print(torch.cuda.device_count())


2
1


In [ ]:
!nvidia-smi

In [ ]:
from VaxMapper.src.llm import load_model_local,build_message
from VaxMapper.src.utils.llm_prompt import CONTRA_EXTRACT_SYSTEM_PROMPT, CONTRA_EXTRACT_USER_PROMPT

In [ ]:
from langgraph.graph import END, StateGraph

In [ ]:
model_id = "google/gemma-4-31B-it"

In [ ]:
# tok = AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

In [ ]:
model = load_model_local(model_id=model_id, device_map="auto")

In [ ]:
generation_config = {
    "temperature": 0.1,         # Low for high precision
    "top_p": 0.8,               # Standard for instruct mode
    "top_k": 20,                # Recommended to prevent language drift
    "repetition_penalty": 1.05, # Prevent MoE looping
    "do_sample": True,          # Required for temperature/top_p/top_k
    "max_new_tokens": 4096,     # Space for clinical data
}

In [ ]:
system_prompt = CONTRA_EXTRACT_SYSTEM_PROMPT
user_prompt_template = CONTRA_EXTRACT_USER_PROMPT
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
from langgraph_agent_runner import load_spl_records_from_file
SPL_LIST_PATH = os.environ.get("SMOKETEST_SPL_LIST", "results/setid_100.txt")
N = 1
all_spl_records = load_spl_records_from_file(SPL_LIST_PATH)
spl_records = all_spl_records[:N]
spl = spl_records[0]
contra_section = extract_section(str(spl.get("SPL_SET_ID")), [CONTRA_Loinc])
result = (contra_section.get("sections")).get(CONTRA_Loinc, {})
contra_text = result.get("section_text")
contra_text

In [ ]:
messages = build_message(system_prompt, user_prompt_template.format(text=contra_text))

In [ ]:
model.generate(messages, **generation_config) 
#6m 7s (qwen 3.5) 

In [ ]:
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes
snomed_frames = load_snomed_dataframes(snomed_source_dir="snomed_us_source")
ff = snomed_frames["snomed_complete_df"]

In [ ]:
ff

In [2]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display
from dotenv import load_dotenv
load_dotenv()

from langgraph_agent_runner import (
    AgentRunConfig,
    ContraLangGraphAgent,
    RunObserver,
    aggregate_agent_results,
    build_llm,
    load_spl_records_from_file,
)


/home/srv-wmanuel3/miniconda3/envs/splmap/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
SPL_LIST_PATH = os.environ.get("SMOKETEST_SPL_LIST", "results/setid_100.txt")
N = 1
BACKEND = os.environ.get("LANGGRAPH_LLM_BACKEND", "huggingface")
MODELID = os.environ.get("HF_MODEL_ID")

print({
    "spl_list_path": SPL_LIST_PATH,
    "n": N,
    "backend": BACKEND,
    "model_id": MODELID,
    "output_dir": os.environ.get("OUTPUT_DIR"),
})

{'spl_list_path': 'results/setid_100.txt', 'n': 1, 'backend': 'huggingface', 'model_id': 'google/gemma-4-31B-it', 'output_dir': 'results/20260415'}


In [4]:
all_spl_records = load_spl_records_from_file(SPL_LIST_PATH)
spl_records = all_spl_records[:N]

# pd.DataFrame(spl_records)

In [5]:
config = AgentRunConfig.from_env()
config

AgentRunConfig(attribute_table={'causative_agent': 246075003, 'severity': 246112005, 'clinical_course': 263502005}, extraction_max_tokens=512, direct_match_max_tokens=56, route_or_fill_max_tokens=128, retries=2, recursion_limit=512, stop=['<<END_JSON>>'], use_strict_prompts=True)

In [ ]:
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

In [ ]:
llm = build_llm(BACKEND)
observer = RunObserver(audit_enabled=False, progress_enabled=True)
config = AgentRunConfig.from_env()
agent = ContraLangGraphAgent(llm=llm, cfg=config, observer=observer)

In [ ]:
config

In [ ]:
result = agent.process_spl(spl_records[0])

In [ ]:
result

In [ ]:
(config.stop,
config.retries,
config.max_llm_tokens_mid,)

In [5]:
spl_records[0]

{'SPL_SET_ID': '083a3282-254f-4042-b0eb-70f76a279f10'}

In [9]:
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
spl = spl_records[0]
contra_section = extract_section(str(spl.get("SPL_SET_ID")), [CONTRA_Loinc])
result = (contra_section.get("sections")).get(CONTRA_Loinc, {})
contra_text = result.get("section_text")
contra_text


'Administration with nitrates and nitric oxide donors (2.4,\n\n4.1)\n\nAdministration with guanylate cyclase (GC) stimulators, such as riociguat (2.4,\n\n4.2)'

In [10]:
contra_section

{'setid': '083a3282-254f-4042-b0eb-70f76a279f10',
 'product_name': 'Vardenafil Hydrochloride',
 'found': True,
 'sections': {'34070-3': {'section_xml': '<section xmlns="urn:hl7-org:v3" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" ID="S4">\n  <id root="8ab5f702-c2d9-430e-b33a-30c9d4798bab"/>\n  <code code="34070-3" codeSystem="2.16.840.1.113883.6.1" displayName="CONTRAINDICATIONS SECTION"/>\n  <title>4 CONTRAINDICATIONS</title>\n  <effectiveTime value="20161103"/>\n  <excerpt>\n    <highlight>\n      <text>\n        <list listType="unordered">\n          <item>Administration with nitrates and nitric oxide donors (\n          \n  \n     <linkHtml href="#S2.4">2.4</linkHtml>, \n          \n  \n     <linkHtml href="#S4.1">4.1</linkHtml>)\n         \n \n    </item>\n          <item>Administration with guanylate cyclase (GC) stimulators, such as riociguat (\n          \n  \n     <linkHtml href="#S2.4">2.4</linkHtml>, \n          \n  \n     <linkHtml href="#S4.2">4.2</linkHtml>)\n   

In [ ]:
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section
from VaxMapper.src.utils.llm_prompt import extract_contraindication_items

max_tokens = 2048
stop = ['<<END_JSON>>']
retries = 2
spl = spl_records[0]
contra_section = extract_section(str(spl.get("SPL_SET_ID")), [CONTRA_Loinc])
result = (contra_section.get("sections")).get(CONTRA_Loinc, {})
contra_text = result.get("section_text")
items, raw = extract_contraindication_items(
    llm.chat,
    contra_text,
    max_tokens=max_tokens,
    stop=stop,
    retries=retries,
)
print("items:", items)
print("raw:", raw)


In [ ]:
# # Things to check 
# 1. max new tokens, max_llm_tokens_mid where do they all connect to in the codebase? Are they being used correctly?
# 2. model kwargs - temp, top_p (qwen has rep and presence penalty, how can we add them as options in the config?)
# 3. does the langgraph agent keep anything in context ? - No. Only state_dict items as defined in the class

In [ ]:
run_rows = []
timings = []

for idx, spl in enumerate(spl_records, start=1):
    started = time.perf_counter()
    result = agent.process_spl(spl)
    elapsed_s = time.perf_counter() - started
    run_rows.append(result)
    timings.append({
        "row": idx,
        "SPL_SET_ID": result.get("SPL_SET_ID"),
        "product_name": result.get("product_name"),
        "contra_section_found": result.get("contra_section_found"),
        "n_items_in": result.get("n_items_in"),
        "n_items_out": result.get("n_items_out"),
        "elapsed_s": round(elapsed_s, 2),
    })

timings_df = pd.DataFrame(timings)
timings_df

In [ ]:
result

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
aggregated_rows, csv_rows = aggregate_agent_results(run_rows)

print({
    "n_run_rows": len(run_rows),
    "n_aggregated_rows": len(aggregated_rows),
    "n_csv_rows": len(csv_rows),
})

In [ ]:
summary_rows = []
for row in run_rows:
    status_counts = {}
    for item in row.get("results", []):
        status = item.get("status", "UNKNOWN")
        status_counts[status] = status_counts.get(status, 0) + 1
    summary_rows.append({
        "SPL_SET_ID": row.get("SPL_SET_ID"),
        "product_name": row.get("product_name"),
        "n_items_in": row.get("n_items_in"),
        "n_items_out": row.get("n_items_out"),
        **status_counts,
    })

summary_df = pd.DataFrame(summary_rows).fillna(0)
summary_df

In [ ]:
csv_df = pd.DataFrame(csv_rows)
csv_df

# medgemma 13
Administration with nitrates
Administration with nitric oxide donors
Administration with guanylate cyclase (GC) stimulators
Administration with riociguat
history of angioedema or hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hereditary or idiopathic angioedema
hereditary angioedema
idiopathic angioedema
co-administer aliskiren with Lisinopril in patients with diabetes
hypersensitivity to any components of the preparation
Known hypersensitivity to aripiprazole (4)
individuals who have shown hypersensitivity to any of its ingredients

# medgemma run 2 (12)
Administration with nitrates
Administration with nitric oxide donors
Administration with guanylate cyclase (GC) stimulators, such as riociguat
history of angioedema or hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hypersensitivity related to previous treatment with an angiotensin converting enzyme inhibitor
hereditary or idiopathic angioedema
hereditary angioedema
idiopathic angioedema
Do not co-administer aliskiren with Lisinopril in patients with diabetes
history of hypersensitivity to any components of the preparation
Known hypersensitivity to aripiprazole (4)
hypersensitivity to any of its ingredients



In [ ]:
problem_rows = []
for row in run_rows:
    for item in row.get("results", []):
        expression = item.get("expression")
        selected_focus_id = item.get("selected_focus_id")
        if expression and isinstance(expression, str) and expression.startswith("N/A"):
            problem_rows.append({
                "SPL_SET_ID": row.get("SPL_SET_ID"),
                "item_index": item.get("item_index"),
                "status": item.get("status"),
                "query_text": item.get("query_text"),
                "selected_focus_id": selected_focus_id,
                "expression": expression,
            })

pd.DataFrame(problem_rows)

In [ ]:
if run_rows:
    display(JSON(run_rows[0], expanded=False))
    display(JSON(aggregated_rows[0], expanded=False))

In [4]:
# 2026-04-03

from agent_runner import (
    AGG_CSV_COLUMNS,
    END as END_MARKER,
    aggregate_agent_results,
    candidate_label_by_id,
    evaluate_aggregated_predictions,
    load_spl_records_from_file,
    parse_json_with_end_marker,
    validate_postcoord_with_mrcm,
    write_csv_rows,
    write_jsonl,
)

In [5]:
import json
# gold_csv = "results/contra_gold_100_2.csv"
gold_csv = "results/contra_gold_100_3.csv"
aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"

In [ ]:
from agent_runner import evaluate_aggregated_predictions
import json
# gold_csv = "results/contra_gold_100_2.csv"
gold_csv = "results/contra_gold_100_3.csv"
aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"
metrics = evaluate_aggregated_predictions(
    pred_csv=aggregated_csv,
    gold_csv=gold_csv,
    out_json=eval_json,
    out_details_csv=eval_details_csv,
    decoupled=True,
    min_pair_score=0.5
)
print(json.dumps(metrics, indent=2))
print(f"Wrote evaluation metrics to: {eval_json}")
print(f"Wrote evaluation details to: {eval_details_csv}")


Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4970.30it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
{
  "inputs": {
    "pred_csv": "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.5,
    "decoupled": true,
    "assignment_method": "hungarian"
  },
  "extraction_level": {
    "tp": 289,
    "fp": 28,
    "fn": 25,
    "precision": 0.9116719242902208,
    "recall": 0.9203821656050956,
    "f1": 0.9160063391442155
  },
  "contraindication_level": {
    "tp": 147,
    "fp": 142,
    "fn": 142,
    "precision": 0.5086505190311419,
    "recall": 0.5086505190311419,
    "f1": 0.5086

In [ ]:
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     | Details
-----------------------------+------------+--------
bert.embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:   can be ignored when loading from different task/architecture; not ok if you expect identical arch.

In [ ]:
# 2026-04-08 
# BM25 index fixes to fix candidate retrueval issues (single word concept missing)

In [19]:
from VaxMapper.src.utils.elastisearch_utils import bulk_index, create_index, get_es_client
from VaxMapper.src.utils.hyb_mapper import SNOMED_CT_SETTINGS, build_es_index
from VaxMapper.src.utils.search_utils import build_snomed_query, bm25_candidates
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes

In [ ]:
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes
snomed_source_dir = "snomed_us_source"
snomed_frames = load_snomed_dataframes(snomed_source_dir=snomed_source_dir)

In [20]:
es = get_es_client()
es.info()
# Returns a space-separated string of index names, then splits into a list
indices = es.cat.indices(h='index', s='index').split()
print(indices)


['rxnorm_bm25', 'snomed_ct_bm25', 'snomed_ct_es_index', 'snomed_ct_es_index_original', 'vo_bm25']


In [ ]:
def get_ancestor_path(concept_id, relationships_df, descriptions_df, max_depth=4, include_concept_id=False):
    path = []
    current = concept_id
    for _ in range(max_depth):
        parent = relationships_df[
            (relationships_df['sourceId'] == current) &
            (relationships_df['typeId'] == 116680003)
        ]['destinationId'].values
        if not len(parent):
            break
        current = parent[0]
        label = descriptions_df[
            descriptions_df['conceptId'] == current
        ]['term'].values[0]
        path.insert(0, label)
    return " → ".join(path)

In [34]:
get_ancestor_path(1269354002, snomed_rel_df, snomed_frames['snomed_complete_df'], max_depth=10)

'SNOMED CT Concept (SNOMED RT+CTV3) → Clinical finding (finding) → Propensity to adverse reaction (finding) → Propensity to adverse reactions to drug (finding)'

In [9]:
import networkx as nx

In [14]:
def get_all_ancestor_paths(concept_id, G, descriptions_df, max_depth=4):
    # BFS edges going "up" (child → parent direction)
    visited_edges = list(nx.bfs_edges(G, concept_id, depth_limit=max_depth))
    
    # Get all reachable ancestors within depth
    reachable = {v for _, v in visited_edges}
    
    # Get all simple paths to each ancestor
    all_paths = []
    for ancestor in reachable:
        for path in nx.all_simple_paths(G, concept_id, ancestor, cutoff=max_depth):
            labels = [
                descriptions_df[descriptions_df['conceptId'] == c]['term'].values[0]
                for c in path
            ]
            print(" → ".join(labels))
            all_paths.append(" → ".join(labels))
            # print(all_paths)
    return all_paths

In [12]:
IS_A = 116680003
G = nx.DiGraph()
is_a = snomed_rel_df[snomed_rel_df['typeId'] == IS_A]
G.add_edges_from(zip(is_a['sourceId'], is_a['destinationId']))  # child → parent

def get_longest_ancestor_path(concept_id, G, descriptions_df, max_depth=10):
    END = list(nx.bfs_edges(G, concept_id, depth_limit=max_depth))[-1][1]
    longest = max(nx.all_simple_paths(G, concept_id, END, cutoff=max_depth), key=len)
    return " → ".join([
        descriptions_df[descriptions_df['conceptId'] == c]['term'].values[0]
        for c in longest
    ])

In [13]:
def get_longest_ancestor_paths(
    concept_id: int,
    is_a_graph: nx.DiGraph,
    concept_meta_df: pd.DataFrame,
    max_depth: int = 10,
) -> str:
    """Return a '→'-joined chain from concept to its most distant IS-A ancestor.

    Uses BFS distance to locate the deepest reachable ancestor within max_depth,
    then returns the single shortest path (= the unique BFS path) to that node.
    Avoids nx.all_simple_paths to prevent exponential blowup on dense hierarchies.
    """
    if concept_id not in is_a_graph:
        return _concept_term(concept_id, concept_meta_df)
    path_lengths = nx.single_source_shortest_path_length(
        is_a_graph, concept_id, cutoff=max_depth
    )
    if len(path_lengths) <= 1:
        return _concept_term(concept_id, concept_meta_df)
    deepest = max(path_lengths, key=path_lengths.get)
    path = nx.shortest_path(is_a_graph, concept_id, deepest)
    return " → ".join(_concept_term(c, concept_meta_df) for c in path)


def _concept_term(concept_id: int, concept_meta_df: pd.DataFrame) -> str:
    """O(1) label lookup from indexed concept_meta_df."""
    try:
        return str(concept_meta_df.at[concept_id, "term"])
    except KeyError:
        return str(concept_id)

In [14]:
get_longest_ancestor_paths(1269354002, G, snomed_frames['snomed_complete_df'], max_depth=10)

'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'

In [6]:
snomed_frames['snomed_complete_df'].set_index('conceptId', inplace=True)

In [7]:
snomed_frames['snomed_complete_df'].at[1269354002, "term"]

'Hypersensitivity to erythromycin (finding)'

In [49]:
_concept_term(1269354002, snomed_frames['snomed_complete_df'])

'1269354002'

In [45]:
get_longest_ancestor_path(1269354002, G, snomed_frames['snomed_complete_df'], max_depth=10)

'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'

In [30]:
root = G.graph.get('root')
root

In [21]:
xx = get_all_ancestor_paths(1269354002, G, snomed_frames['snomed_complete_df'], max_depth=5)

Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Hypersensitivity condition (finding) → Clinical finding (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Hypersensitivity condition (finding)
Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding)
Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding)
Hypersensitivity to erythromycin (finding) → Propensi

In [20]:
xx

['Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Hypersensitivity condition (finding)',
 'Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding)',
 'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding)',
 'Hypersensitivity to erythromycin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding)',
 'Hypersensitivity to erythromycin (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding)']

In [23]:
longest = max(nx.all_simple_paths(G,293619005, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))

Allergy to ibuprofen (finding) → Allergy to non-steroidal anti-inflammatory agent (finding) → Allergy to drug (finding) → Allergic disposition (finding) → Allergic condition (finding) → Hypersensitivity condition (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [24]:
longest = max(nx.all_simple_paths(G,1269353008, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))

Hypersensitivity to gentamicin (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [25]:
longest = max(nx.all_simple_paths(G,218613000, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))


Adverse reaction caused by ibuprofen (disorder) → Non-steroidal anti-inflammatory drug adverse reaction (disorder) → Non-opioid analgesic adverse reaction (disorder) → Analgesic adverse reaction (disorder) → Adverse reaction caused by drug (disorder) → Adverse reaction (disorder) → Disease (disorder) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [26]:
longest = max(nx.all_simple_paths(G,1269400003, 138875005), key=len)
print(" → ".join([snomed_frames['snomed_complete_df'][snomed_frames['snomed_complete_df']['conceptId'] == c]['term'].values[0] for c in longest]))


Hypersensitivity to nalidixic acid (finding) → Hypersensitivity disposition (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)


In [43]:
visited_edges = list(nx.bfs_edges(G, 1269354002, depth_limit=10))
visited_edges

[(1269354002, 419511003),
 (1269354002, 609433001),
 (419511003, 420134006),
 (609433001, 473010000),
 (420134006, 404684003),
 (404684003, 138875005)]

In [44]:
reachable = {v for _, v in visited_edges}
reachable

{138875005, 404684003, 419511003, 420134006, 473010000, 609433001}

In [46]:
list(nx.all_simple_paths(G, 1269354002, 138875005, cutoff=10))

[[1269354002, 419511003, 420134006, 404684003, 138875005],
 [1269354002, 609433001, 420134006, 404684003, 138875005],
 [1269354002, 609433001, 473010000, 404684003, 138875005]]

In [15]:
get_ancestor_path(1269354002, snomed_rel_df, snomed_frames["snomed_complete_df"], max_depth=10)

NameError: name 'get_ancestor_path' is not defined

In [18]:
SNOMED_CT_SETTINGS

{'settings': {'index': {'number_of_shards': 1,
   'number_of_replicas': 1,
   'similarity': {'default': {'type': 'BM25', 'k1': 1.2, 'b': 0.5}}},
  'analysis': {'filter': {'snomed_shingle:': {'type': 'shingle',
     'min_shingle_size': 2,
     'max_shingle_size': 3,
     'output_unigrams': True}},
   'analyzer': {'snomed_text': {'type': 'custom',
     'tokenizer': 'standard',
     'filter': ['lowercase', 'asciifolding']},
    'snomed_text_shingle': {'type': 'custom',
     'tokenizer': 'standard',
     'filter': ['lowercase', 'asciifolding', 'snomed_shingle:']}}}},
 'mappings': {'properties': {'conceptId': {'type': 'keyword'},
   'preferredTerm': {'type': 'text',
    'analyzer': 'snomed_text',
    'search_analyzer': 'snomed_text',
    'copy_to': ['all_terms'],
    'fields': {'exact': {'type': 'keyword'},
     'phrase': {'type': 'text',
      'analyzer': 'snomed_text',
      'index_options': 'offsets'},
     'shingle': {'type': 'text', 'analyzer': 'snomed_text_shingle'}}},
   'synonyms': 

In [5]:
es_index= "snomed_ct_bm25"
build_es_index(es,es_index,snomed_frames["snomed_complete_df"],rebuild_index=True, index_settings=SNOMED_CT_SETTINGS)

Deleting existing index: snomed_ct_bm25
Creating index 'snomed_ct_bm25'...


In [19]:
es.info()

ObjectApiResponse({'name': 'msdplaplp001.uth.tmc.edu', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'm4zPSJyjTKCUH2PiE0ADfw', 'version': {'number': '9.0.0', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '112859b85d50de2a7e63f73c8fc70b99eea24291', 'build_date': '2025-04-08T15:13:46.049795831Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

In [20]:
es_index = 'snomed_ct_es_index'
# 'snomed_ct_es_index_original

In [ ]:
settings = es.indices.get_settings(index="snomed_ct_es_index")
print(settings)

{'snomed_ct_es_index': {'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'number_of_shards': '1', 'provided_name': 'snomed_ct_es_index', 'similarity': {'default': {'type': 'BM25', 'b': '0.5', 'k1': '1.2'}}, 'creation_date': '1776198453404', 'analysis': {'filter': {'snomed_shingle:': {'max_shingle_size': '3', 'min_shingle_size': '2', 'output_unigrams': 'true', 'type': 'shingle'}}, 'analyzer': {'snomed_text_shingle': {'filter': ['lowercase', 'asciifolding', 'snomed_shingle:'], 'type': 'custom', 'tokenizer': 'standard'}, 'snomed_text': {'filter': ['lowercase', 'asciifolding'], 'type': 'custom', 'tokenizer': 'standard'}}}, 'number_of_replicas': '1', 'uuid': 'dug0fIVpQ9eHyCFz_ERREg', 'version': {'created': '9009000'}}}}}


In [22]:
settings = es.indices.get_settings(index="snomed_ct_es_index_original")
print(settings)

{'snomed_ct_es_index_original': {'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'number_of_shards': '1', 'provided_name': 'snomed_ct_es_index_original', 'similarity': {'default': {'type': 'BM25', 'b': '0.75', 'k1': '1.2'}}, 'creation_date': '1776185692830', 'analysis': {'filter': {'snomed_shingle:': {'max_shingle_size': '3', 'min_shingle_size': '2', 'output_unigrams': 'true', 'type': 'shingle'}}, 'analyzer': {'snomed_text_shingle': {'filter': ['lowercase', 'asciifolding', 'snomed_shingle:'], 'type': 'custom', 'tokenizer': 'standard'}, 'snomed_text': {'filter': ['lowercase', 'asciifolding'], 'type': 'custom', 'tokenizer': 'standard'}}}, 'number_of_replicas': '1', 'uuid': 'N1Ms1NFVTtKLA2Rz93j1CA', 'version': {'created': '9009000'}}}}}


In [8]:
mappings = es.indices.get_mapping(index=es_index)
print(mappings)

{'snomed_ct_bm25': {'mappings': {'properties': {'all_terms': {'type': 'text', 'analyzer': 'snomed_text'}, 'conceptId': {'type': 'keyword'}, 'preferredTerm': {'type': 'text', 'fields': {'exact': {'type': 'keyword'}, 'phrase': {'type': 'text', 'index_options': 'offsets', 'analyzer': 'snomed_text'}, 'shingle': {'type': 'text', 'analyzer': 'snomed_text_shingle'}}, 'copy_to': ['all_terms'], 'analyzer': 'snomed_text'}, 'semantic_tag': {'type': 'keyword'}, 'synonyms': {'type': 'text', 'copy_to': ['all_terms'], 'analyzer': 'snomed_text'}}}}}


In [33]:
query_text = "Mycobacterial infection of the eye"
# Old call — still works, no changes needed
results = bm25_candidates(
    es=es, query_text=query_text, index="snomed_ct_es_index", k=50,
      text_field="all_terms", id_field="conceptId", label_field="preferredTerm")
[r['label'] for r in results]

['Mycobacterial infection of the central nervous system (disorder)',
 'Neonatal infection of the eye (disorder)',
 'Atypical mycobacterial infection of lung (disorder)',
 'Atypical mycobacterial infection of hand (disorder)',
 'Atypical mycobacterial infection (disorder)',
 'Disseminated atypical mycobacterial infection (disorder)',
 'Infection of tendon sheath caused by Mycobacterium (disorder)',
 'Keratitis of right eye caused by Mycobacterium (disorder)',
 'Keratitis of left eye caused by Mycobacterium (disorder)',
 'Mycobacterial infection (excluding tuberculosis AND leprosy) (disorder)',
 'Skin and soft tissue atypical mycobacterial infection (disorder)',
 'Mendelian susceptibility to mycobacterial disease (disorder)',
 'Susceptibility to viral and mycobacterial infection (disorder)',
 'Gonococcal infection of eye (disorder)',
 'Massaging of the eye (procedure)',
 'At increased risk for infection of eye (finding)',
 'Bacterial infection of the nervous system (disorder)',
 'Bacteri

In [ ]:
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes
snomed_source_dir = "snomed_us_source"
snomed_frames = load_snomed_dataframes(snomed_source_dir=snomed_source_dir)

In [6]:
from VaxMapper.src.utils.search_utils import search_query

/home/srv-wmanuel3/miniconda3/envs/splmap/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [8]:
import torch

if torch.cuda.is_available():
    print(f"Number of visible GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No CUDA GPUs available.")

Number of visible GPUs: 1
GPU 0: NVIDIA A100-SXM4-80GB


In [11]:

# from sentence_transformers import CrossEncoder
model_name = "google/embeddinggemma-300m"
# reranker_model_id = "cross-encoder/ms-marco-MiniLM-L6-v2"
device="cuda:0"
dense_index_path_old = "FAISS/embeddinggemma-300m.bin"
dense_index_path_new = "FAISS/enriched_embeddinggemma-300m.bin"
# st_model = load_ST_model(model_name, device="cuda:0")
# cross_encoder = CrossEncoder(reranker_model_id, device="cuda:0")

In [5]:
import pandas as pd
from typing import Optional, Sequence, Set, Dict, List
ISA = 116680003
from collections import defaultdict
import re
def create_enriched_terms_df(
    snomed_complete_df: pd.DataFrame,
    rel_df: pd.DataFrame,
    exclude_type_ids: Optional[Set[int]] = None,
) -> pd.DataFrame:
    """
    Build one row per concept with a rich pipe-delimited embedding string.

    Format:
        preferred_term | syn1 | syn2 | ... | semantic_tag | attr_name1 = val1 | ...

    Columns returned:
        - conceptId  (int)
        - term_text  (str)
    """
    if exclude_type_ids is None:
        exclude_type_ids = {int(ISA)}  # always exclude IS-A

    # Build a fast lookup: conceptId -> preferred term (stripped of semantic tag)
    id_to_term: Dict[int, str] = {}
    for _, row in snomed_complete_df.iterrows():
        cid = int(row["conceptId"])
        # Strip semantic tag from preferred term for attribute names/values
        id_to_term[cid] = re.sub(r"\s*\([^)]+\)\s*$", "", str(row["term"])).strip()

    # Filter relationships: active, non-excluded types only
    attr_df = rel_df[~rel_df["typeId"].isin(exclude_type_ids)].copy()

    # Group attributes per source concept
    attr_map: Dict[int, List[str]] = defaultdict(list)
    for _, row in attr_df.iterrows():
        src = int(row["sourceId"])
        type_name = id_to_term.get(int(row["typeId"]), str(row["typeId"]))
        dest_name = id_to_term.get(int(row["destinationId"]), str(row["destinationId"]))
        attr_map[src].append(f"{type_name} = {dest_name}")

    rows = []
    for _, row in snomed_complete_df.iterrows():
        cid = int(row["conceptId"])
        parts = [str(row["term"])]                          # preferred term (with tag)
        synonyms = row.get("synonyms") or []
        parts.extend(str(s) for s in synonyms if s)        # all synonyms
        parts.append(str(row.get("semantic_tag", "")))      # semantic tag
        parts.extend(attr_map.get(cid, []))                 # attribute = value pairs
        term_text = " | ".join(p for p in parts if p)
        rows.append({"conceptId": cid, "term_text": term_text})

    return pd.DataFrame(rows)

In [9]:
en_terms_df = create_enriched_terms_df(
    snomed_complete_df=snomed_frames["snomed_complete_df"],
    rel_df=snomed_frames["rel_df"],
    exclude_type_ids={ISA},  # only exclude IS-A for embeddings
)
en_terms_df.head()

,conceptId,term_text
0,127362006,Previous pregnancies (finding) | Previous preg...
1,125587004,Superficial injury (disorder) | Superficial in...
2,125643001,Open wound (disorder) | Open wound | Open woun...
3,125665001,Crushing injury (disorder) | Crushing injury |...
4,125666000,Burn (disorder) | Burn | disorder | Associated...


In [12]:
from VaxMapper.src.utils.hyb_mapper import build_or_load_faiss_index
st_model, faiss_index_old = build_or_load_faiss_index(
        terms_df=snomed_frames["terms_df"],
        model_name=model_name,
        device=device,
        index_path=dense_index_path_old,
        rebuild_index=False,
    )

Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 5459.00it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Moving FAISS index to GPU (first visible device, FAISS device 0)...
Index moved to GPU successfully.


In [13]:
st_model, faiss_index_new = build_or_load_faiss_index(
        terms_df=en_terms_df,
        model_name=model_name,
        device=device,
        index_path=dense_index_path_new,
        rebuild_index=False,
    )

Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 5851.07it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Moving FAISS index to GPU (first visible device, FAISS device 0)...
Index moved to GPU successfully.


In [14]:
concept_meta_df = snomed_frames["concept_df"].set_index("conceptId")

In [98]:
query_text_original = "Mycobacterial infection of the eye"
query_text_serialized = "Mycobacterial infection of the eye | Pathological process = infectious process | Causative agent = Mycobacterium | Finding site = eye"

In [28]:
query_text_serialized = "Urticaria after taking NSAIDs| pathological process =Urticaria  substance_text=NSAID"

In [29]:
res = search_query(query_text=query_text_serialized,model=st_model,faiss_index=faiss_index_old,concept_meta_df=concept_meta_df,es=es,label_column="term",bm25_index="snomed_ct_es_index",bm25_text_field="preferredTerm",bm25_id_field="conceptId",bm25_label_field="preferredTerm",k_bm25=50,k_dense=50,k_final=25,n_final=15,
# cross_encoder=cross_encoder,
)
print(f"Final candidates: {[(r['label']) for r in res]}")

Final candidates: ['Non-steroid anti-inflammatory drug-induced angioedema-urticaria (disorder)', 'Urticaria medicamentosa (disorder)', 'Idiopathic urticaria (disorder)', 'Drug-aggravated angioedema-urticaria (disorder)', 'Allergic reaction caused by nonsteroidal antiinflammatory agent (disorder)', 'Bullous urticaria (disorder)', 'Allergy to non-steroidal anti-inflammatory agent (finding)', 'Photoallergic contact dermatitis due to non-steroidal anti-inflammatory drug (disorder)', 'Pressure urticaria (disorder)', 'Analgesics and non-steroidal anti-inflammatory drug allergy (navigational concept)', 'Urticaria (disorder)', 'Anaphylaxis caused by non-steroidal anti-inflammatory agent (disorder)', 'Episodic urticaria (disorder)', 'Non-allergic hypersensitivity to non-steroidal anti-inflammatory agent (finding)', 'Solar urticaria (disorder)']


In [30]:
res = search_query(query_text=query_text_serialized,model=st_model,faiss_index=faiss_index_new,concept_meta_df=concept_meta_df,es=es,label_column="term",bm25_index="snomed_ct_es_index_original",bm25_text_field="preferredTerm",bm25_id_field="conceptId",bm25_label_field="preferredTerm",k_bm25=50,k_dense=50,k_final=25,n_final=15,# cross_encoder=cross_encoder,
                   )
print(f"Final candidates: {[(r['label']) for r in res]}")

Final candidates: ['Urticaria medicamentosa (disorder)', 'Idiopathic urticaria (disorder)', 'Allergic urticaria (disorder)', 'Anaphylactic urticaria (disorder)', 'Non-steroid anti-inflammatory drug-induced angioedema-urticaria (disorder)', 'Urticaria (disorder)', 'Allergic reaction caused by nonsteroidal antiinflammatory agent (disorder)', 'Bullous urticaria (disorder)', 'Allergy to non-steroidal anti-inflammatory agent (finding)', 'Episodic urticaria (disorder)', 'Anaphylaxis caused by non-steroidal anti-inflammatory agent (disorder)', 'Pressure urticaria (disorder)', 'Non-allergic anaphylaxis caused by non steroidal anti-inflammatory drug (disorder)', 'Chronic urticaria (disorder)', 'Non-allergic hypersensitivity to non-steroidal anti-inflammatory agent (finding)']


In [34]:
# New call — uses multi-tier query
custom_query = build_snomed_query(
    query_text,
    text_field="preferredTerm",
    # semantic_tags=["disorder", "finding"],  # optional, pass None to skip
    semantic_tags=None 
)
results_new = bm25_candidates(
    es,
    query_text,
    index="snomed_ct_es_index",
    k=50,
    id_field="conceptId",
    label_field="preferredTerm",
    custom_query=custom_query,
)
[r['label'] for r in results_new]

['Mycobacterial infection of the central nervous system (disorder)',
 'Neonatal infection of the eye (disorder)',
 'Atypical mycobacterial infection of lung (disorder)',
 'Atypical mycobacterial infection of hand (disorder)',
 'Atypical mycobacterial infection (disorder)',
 'Disseminated atypical mycobacterial infection (disorder)',
 'Mycobacterial infection (excluding tuberculosis AND leprosy) (disorder)',
 'Susceptibility to viral and mycobacterial infection (disorder)',
 'Skin and soft tissue atypical mycobacterial infection (disorder)',
 'Massaging of the eye (procedure)',
 'Bacterial infection of the nervous system (disorder)',
 'Bacterial infection of the digestive tract (disorder)',
 'Viral infection of the digestive tract (disorder)',
 'Streptococcus group B infection of the infant (disorder)',
 'Brucella infection of the central nervous system (disorder)',
 'Chlamydial infection of the central nervous system (disorder)',
 'Coccidioides infection of the central nervous system (

In [30]:
custom_query

{'bool': {'should': [{'term': {'preferredTerm.exact': {'value': 'mycobacterial infection of the eye',
      'boost': 5.0}}},
   {'match_phrase': {'preferredTerm.phrase': {'query': 'Mycobacterial infection of the eye',
      'boost': 3.0}}},
   {'match': {'preferredTerm.shingle': {'query': 'Mycobacterial infection of the eye',
      'boost': 2.0}}},
   {'match': {'all_terms': {'query': 'Mycobacterial infection of the eye',
      'operator': 'or',
      'boost': 0.6}}}],
  'minimum_should_match': 1}}

In [22]:
from VaxMapper.src.utils.hyb_mapper import (
    DEFAULT_ITEM_TERM_KEYS,
    get_cached_mapper_resources,
    retrieve_candidates_for_item as retrieve_candidates_for_item_hybrid,
    map_item_terms
)

In [17]:
from langgraph_agent_runner import retrieve_candidates_for_item
import os

In [18]:
resources = get_cached_mapper_resources(
    snomed_source_dir=os.environ.get("SNOMED_SOURCE_DIR", "snomed_us_source"),
    concept_path=os.environ.get("SNOMED_CONCEPT_PATH"),
    description_path=os.environ.get("SNOMED_DESCRIPTION_PATH"),
    es_index=os.environ.get("MAPPER_ES_INDEX", "snomed_ct_es_index"),
    dense_index_path=os.environ.get("MAPPER_DENSE_INDEX_PATH", "results/snomed_terms_dense_test.bin"),
    model_name=os.environ.get("MAPPER_MODEL_NAME", "tavakolih/all-MiniLM-L6-v2-pubmed-full"),
    device=os.environ.get("MAPPER_DEVICE", "cuda:0"),
    k_dense=int(os.environ.get("MAPPER_K_DENSE", "50")),
    k_bm25=int(os.environ.get("MAPPER_K_BM25", "50")),
    k_final=int(os.environ.get("MAPPER_K_FINAL", "20")),
    rebuild_dense_index=os.environ.get("MAPPER_REBUILD_DENSE_INDEX", "").lower() in {"1", "true", "yes"},
    rebuild_es_index=os.environ.get("MAPPER_REBUILD_ES_INDEX", "").lower() in {"1", "true", "yes"},
    item_term_keys=DEFAULT_ITEM_TERM_KEYS,
    reranker_model_id=os.environ.get("RERANKER_MODEL_ID", ""),
    reranker_device=os.environ.get("RERANKER_DEVICE", "cuda:0"),
)

Index 'snomed_ct_bm25' already exists; skipping creation.
Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7629.45it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Moving FAISS index to GPU (first visible device, FAISS device 0)...
Index moved to GPU successfully.


In [25]:
type(resources)

VaxMapper.src.utils.hyb_mapper.MapperResources

In [65]:
snomed_frames.keys()

dict_keys(['concept_df', 'synonym_df', 'terms_df', 'snomed_complete_df', 'domain', 'attr_domain', 'attr_range', 'rel_df'])

In [63]:
snomed_frames["terms_df"]

,conceptId,term_text,term_type
0,127362006,Previous pregnancies (finding),preferred
1,125587004,Superficial injury (disorder),preferred
2,125643001,Open wound (disorder),preferred
3,125665001,Crushing injury (disorder),preferred
4,125666000,Burn (disorder),preferred
...,...,...,...
1017041,900000000000548007,Preferred,synonym
1017042,900000000000549004,Acceptable,synonym
1017043,900000000000550004,Definition,synonym
1017044,625544171000132109,Disorder of pancreatic stent,synonym


In [91]:
en_terms_df[en_terms_df["conceptId"]==15991511000119102]["term_text"].values[0]

'Chorioretinitis of right eye caused by Mycobacterium tuberculosis (disorder) | Chorioretinitis of right eye caused by Mycobacterium tuberculosis | Right chorioretinitis caused by Mycobacterium tuberculosis | disorder | Pathological process = Infectious process | Causative agent = Mycobacterium tuberculosis | Pathological process = Infectious process | Finding site = Structure of choroid of right eye | Finding site = Structure of retina of right eye | Causative agent = Mycobacterium tuberculosis | Associated morphology = Inflammatory morphology | Associated morphology = Inflammatory morphology'

In [23]:
item = {
    "SPL_SET_ID": "test_spl",
    "item_index": 0,
    "ci_text": "hypersensitivity disorder"}

In [24]:
retrieve_candidates_for_item_hybrid(item, resources)

{'focus_candidates': [{'id': 426760008,
   'label': 'Delayed hypersensitivity disorder (disorder)',
   'fused': 0.032266458495966696,
   'from': 'bm25,dense',
   'ancestor_path': 'Delayed hypersensitivity disorder (disorder) → Immune hypersensitivity disorder by mechanism (disorder) → Hypersensitivity condition (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'},
  {'id': 421961002,
   'label': 'Hypersensitivity reaction (disorder)',
   'fused': 0.030776515151515152,
   'from': 'bm25,dense',
   'ancestor_path': 'Hypersensitivity reaction (disorder) → Hypersensitivity condition (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)'},
  {'id': 21626009,
   'label': 'Cutaneous hypersensitivity (disorder)',
   'fused': 0.03057889822595705,
   'from': 'bm25,dense',
   'ancestor_path': 'Cutaneous hypersensitivity (disorder) → Disorder of skin (disorder) → Disorder of skin and/or subcutaneous tissue (disorder) → Disorder of integument (disorder)

In [26]:
import json


def get_available_keys(audit_file, spl_set_id, item_index, call_name):
    """Return all keys present in the matching llm_call record.

    Matches on spl_set_id, item_index, and call_name.
    Returns an empty list if no matching record is found.
    """
    with open(audit_file) as f:
        for line in f:
            record = json.loads(line)
            if (
                record.get("event_type") == "llm_call"
                and record.get("spl_set_id") == spl_set_id
                and record.get("item_index") == item_index
                and record.get("call_name") == call_name
            ):
                return list(record.keys())
    return []


def get_values(audit_file, spl_set_id, item_index, call_name, keys):
    """Return a dict of {key: value} for the requested keys from the matching llm_call record.

    Matches on spl_set_id, item_index, and call_name.
    Returns an empty dict if no matching record is found.
    Keys absent from the record are returned with value None.
    """
    with open(audit_file) as f:
        for line in f:
            record = json.loads(line)
            if (
                record.get("event_type") == "llm_call"
                and record.get("spl_set_id") == spl_set_id
                and record.get("item_index") == item_index
                and record.get("call_name") == call_name
            ):
                return {k: record.get(k) for k in keys}
    return {}


### Improvement by prompt examples for Extraction Stage

In [ ]:
## 0.5 CI 0.6 concept F1 - Baseline
AUDIT = "results/20260406/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 0
call_name = "direct_match"
available = get_available_keys(AUDIT, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt"])
print("values:", vals)

# "parsed_output", "parse_success", "duration_s"

values: {'system_prompt': '\nYou are a strict biomedical terminology validator.\nYour role is to determine if a candidate is a LEXICAL and SEMANTIC identity match for the contraindication query.\n\nYour ONLY task:\n- Identify if a DIRECT MATCH exists among the provided candidates.\n- If a match exists, select exactly ONE candidate from the list.\n- Otherwise, return no match.\n\n--------------------\nDIRECT MATCH (STRICT)\n--------------------\nA candidate is a DIRECT MATCH only if it is an exact semantic equivalent. Do NOT bridge concepts even if they are clinically related.\n\n1) Lexical-Semantic Alignment:\n- The candidate must match the specificity and naming of the query.\n- Do NOT bridge concepts that use different primary terms even if they are clinically related.\n- If the query text and candidate label belong to different levels of the hierarchy, return no match.\n2) Temporal/Contextual Scope:\n- "Post-X" is NOT a match for "X".\n- "History of X" is NOT a match for "X".\n- If 

values: {'user_prompt': 'QUERY:\n"Mycobacterial infection of the eye"\n\nCANDIDATES (choose from these only):\n1) 866083008 | Concept name: Infection of eyelid caused by Mycobacterium tuberculosis (disorder) | Score: 6.615\n2) 1163500007 | Concept name: Infection of eyelid caused by Mycobacterium leprae (disorder) | Score: 6.145\n3) 15693481000119109 | Concept name: Keratitis of bilateral eyes caused by Mycobacterium (disorder) | Score: 3.954\n4) 128984004 | Concept name: Bacterial infection of eye (disorder) | Score: 3.845\n5) 15693521000119109 | Concept name: Keratitis of left eye caused by Mycobacterium (disorder) | Score: 3.331\n6) 15693441000119104 | Concept name: Keratitis of right eye caused by Mycobacterium (disorder) | Score: 3.331\n7) 406594003 | Concept name: Mycobacterial infection of the central nervous system (disorder) | Score: 2.872\n8) 35876006 | Concept name: Gonococcal infection of eye (disorder) | Score: 2.826\n9) 111811007 | Concept name: Mycobacterial infection (e

In [ ]:
## 0.6 CI 0.7 concept F1 - Extraction Prompt Update with example
AUDIT_2 = "results/20260407/gemma4/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 0
call_name = "direct_match"
available = get_available_keys(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt"])
print("values:", vals)


values: {'system_prompt': '\nYou are a strict biomedical terminology validator.\nYour role is to determine if a candidate is a LEXICAL and SEMANTIC identity match for the contraindication query.\n\nYour ONLY task:\n- Identify if a DIRECT MATCH exists among the provided candidates.\n- If a match exists, select exactly ONE candidate from the list.\n- Otherwise, return no match.\n\nWhen in doubt, return no match. A false negative here is recoverable; a false positive is not.\n\n--------------------\nDIRECT MATCH (STRICT)\n--------------------\nA candidate is a DIRECT MATCH only if its label is an exact lexical-semantic equivalent to the query — identical clinical meaning, identical ontological level, identical primary term.\n\n1) Primary Term Identity:\n- The root condition word used in the query MUST be the same root condition word in the candidate label.\n- Clinically related terms that are NOT interchangeable in SNOMED CT:\n    * "Hypersensitivity" ≠ "Allergy"  (Allergy is a subtype of

### Improvement by BM25 tuning

In [34]:
## 0.6 CI 0.7 concept F1 - Extraction Prompt Update with example
AUDIT_2 = "results/20260407/gemma4/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 6
call_name = "direct_match"
available = get_available_keys(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)

values: {'user_prompt': 'QUERY:\n"allergic-type reactions after taking other NSAIDs"\n\nCANDIDATES (choose from these only):\n1) 293610009 |Allergy to non-steroidal anti-inflammatory agent (finding)|\n2) 871931002 |Allergic reaction after allergen immunotherapy (disorder)|\n3) 15920041000119105 |Allergic reaction caused by nonsteroidal antiinflammatory agent (disorder)|\n4) 386665002 |Pupil reactions normal (finding)|\n5) 293581006 |Analgesics and non-steroidal anti-inflammatory drug allergy (navigational concept)|\n6) 396079007 |Assessment of adverse drug reactions (procedure)|\n7) 781684006 |Non-allergic hypersensitivity to non-steroidal anti-inflammatory agent (finding)|\n8) 370899004 |Retention of foreign object in a patient after surgery or other procedure (event)|\n9) 293619005 |Allergy to ibuprofen (finding)|\n10) 419511003 |Propensity to adverse reactions to drug (finding)|\n', 'parsed_output': {'direct_match': True, 'selected_id': '15920041000119105', 'selected_term': 'Allergi

In [33]:
## 0.4 CI 0.6 concept F1 - Extraction Prompt Update with example + BM25 tuning
AUDIT_3 = "results/20260407/gemma4_bm25/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"

# SPL-level extraction call (item_index is None)
item_index = 6
call_name = "direct_match"
available = get_available_keys(AUDIT_3, SET_ID, item_index=item_index, call_name=call_name)
# print("available keys:", available)

vals = get_values(AUDIT_3, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)

values: {'user_prompt': 'QUERY:\n"allergic-type reactions after taking other NSAIDs"\n\nCANDIDATES (choose from these only):\n1) 293610009 |Allergy to non-steroidal anti-inflammatory agent (finding)|\n2) 15920041000119105 |Allergic reaction caused by nonsteroidal antiinflammatory agent (disorder)|\n3) 293581006 |Analgesics and non-steroidal anti-inflammatory drug allergy (navigational concept)|\n4) 781684006 |Non-allergic hypersensitivity to non-steroidal anti-inflammatory agent (finding)|\n5) 293619005 |Allergy to ibuprofen (finding)|\n6) 293625009 |Allergy to naproxen (finding)|\n7) 419901001 |Non-steroidal anti-inflammatory drug adverse reaction (disorder)|\n8) 15919911000119108 |Allergic reaction caused by analgesic (disorder)|\n9) 416093006 |Allergic reaction caused by drug (disorder)|\n10) 293584003 |Allergy to paracetamol (finding)|\n', 'parsed_output': {'direct_match': True, 'selected_id': '15920041000119105', 'selected_term': 'Allergic reaction caused by nonsteroidal antiinfla

In [38]:
## 0.6 CI 0.7 concept F1 - Extraction Prompt Update with example
AUDIT_2 = "results/20260407/gemma4/runtime_audit.jsonl"
## 0.4 CI 0.6 concept F1 - Extraction Prompt Update with example + BM25 tuning
AUDIT_3 = "results/20260407/gemma4_bm25/runtime_audit.jsonl"
SET_ID = "2b470e48-c23c-4f5c-916a-d38d8518896b"


item_index = 0
call_name = "route_or_fill"

for audit in [AUDIT_2, AUDIT_3]:
    vals = get_values(audit, SET_ID, item_index=item_index, call_name=call_name,
                      keys=["system_prompt","user_prompt","parsed_output"])
    print("values:", vals)
    print("\n\n")

values: {'system_prompt': '\nYou are a SNOMED CT minimal representation assistant for contraindications.\n\nYour job has two parts:\n1) Decide whether the contraindication can be sufficiently represented using ONLY:\n   - one focus concept\n   - optional causative_agent\n   - optional severity\n   - optional clinical_course\n2) Regardless of that decision, extract the best available minimal concept representation from the provided candidates.\n\nRules:\n- Always select the best focus concept you can from FOCUS_CANDIDATES, unless no credible focus exists.\n- Always select the best value for each attribute from its candidate list when supported by the text.\n- If no supported value exists for an attribute, output "N/A" for that attribute.\n- Use ONLY IDs from the provided candidate lists.\n- Do NOT invent IDs.\n\npost_decision rules:\n- "YES" if the contraindication is sufficiently represented by the minimal model above.\n- "N/A" if clinically important meaning remains outside that model

### Isolation Run - Issues between the enhanced and simple dense index


In [45]:
AUDIT_1 = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
AUDIT_2 = "results/20260415_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "6ae413ba-5a48-423f-8b8d-7a42661cf836"
item_index = 0
call_name = "direct_match"
print(f"Are system prompts identical? {get_values(AUDIT_1, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt') == get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt')}")
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Simple Dense Index:", vals,)
print("\n------------------------------------------")
vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Enhanced Dense Index:", vals)
# print("\n------------------------------------------")
# call_name = "route_or_fill"
# vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
#                   keys=["user_prompt","parsed_output"])
# print("values:", vals)

Are system prompts identical? True
Simple Dense Index: {'user_prompt': 'QUERY:\n"Known hypersensitivity to naproxen"\n\nCANDIDATES (choose from these only):\n1) 609534005 | Concept name: Non-allergic hypersensitivity to losartan (finding) | Score: -1.199 | Ancestor path: Non-allergic hypersensitivity to losartan (finding) → Non-allergic hypersensitivity to angiotensin II receptor antagonist (finding) → Non-allergic hypersensitivity to drug or medicament (finding) → Propensity to adverse reactions to drug (finding) → Propensity to adverse reaction (finding) → Clinical finding (finding) → SNOMED CT Concept (SNOMED RT+CTV3)\n2) 218620007 | Concept name: Adverse reaction caused by naproxen (disorder) | Score: -1.312 | Ancestor path: Adverse reaction caused by naproxen (disorder) → Non-steroidal anti-inflammatory drug adverse reaction (disorder) → Non-opioid analgesic adverse reaction (disorder) → Analgesic adverse reaction (disorder) → Adverse reaction caused by drug (disorder) → Adverse r

In [46]:
AUDIT_1 = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
AUDIT_2 = "results/20260415_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "a5aaf1db-1938-4073-9da4-c47f959d013e"
item_index = 8
call_name = "direct_match"
print(f"Are system prompts identical? {get_values(AUDIT_1, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt') == get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt')}")
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Simple Dense Index:", vals)
print("\n------------------------------------------")
vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Enhanced Dense Index:", vals)
# print("\n------------------------------------------")
# call_name = "route_or_fill"
# vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
#                   keys=["user_prompt","parsed_output"])
# print("values:", vals)

Are system prompts identical? True
Simple Dense Index: {'user_prompt': 'QUERY:\n"intestinal atony"\n\nCANDIDATES (choose from these only):\n1) 29479008 | Concept name: Atony of colon (disorder) | Score: 2.474 | Ancestor path: Atony of colon (disorder) → Disorder of colon (disorder) → Disorder of large intestine (disorder) → Disorder of lower gastrointestinal tract (disorder) → Disorder of abdominopelvic segment of trunk (disorder) → Disorder of trunk (disorder) → Finding of trunk structure (finding)\n2) 432414001 | Concept name: Constipation due to atony of colon (disorder) | Score: 1.013 | Ancestor path: Constipation due to atony of colon (disorder) → Functional disorder of intestine (disorder) → Disorder of intestine (disorder) → Disorder of abdomen (disorder) → Finding of abdomen (finding) → Finding of abdominopelvic segment of trunk (finding) → Finding of trunk structure (finding)\n3) 427225008 | Concept name: Atony of esophagus (disorder) | Score: 0.624 | Ancestor path: Atony of e

In [47]:
AUDIT_1 = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
AUDIT_2 = "results/20260415_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "a5aaf1db-1938-4073-9da4-c47f959d013e"
item_index = 8
call_name = "route_or_fill"
print(f"Are system prompts identical? {get_values(AUDIT_1, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt') == get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt')}")
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Simple Dense Index:", vals)
print("\n------------------------------------------")
vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Enhanced Dense Index:", vals)
# print("\n------------------------------------------")
# call_name = "route_or_fill"
# vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
#                   keys=["user_prompt","parsed_output"])
# print("values:", vals)

Are system prompts identical? True
Simple Dense Index: {'user_prompt': 'QUERY:\nintestinal atony\n\nEXTRACTED_FIELDS:\ncontraindication_state_text=intestinal atony\nsubstance_text=None\nseverity_span=None\ncourse_span=None\n\nATTRIBUTE_TABLE:\n{"causative_agent":246075003,"severity":246112005,"clinical_course":263502005}\n\nFOCUS_CANDIDATES:\n1) 29479008 | Concept name: Atony of colon (disorder) | Score: 2.474 | Ancestor path: Atony of colon (disorder) → Disorder of colon (disorder) → Disorder of large intestine (disorder) → Disorder of lower gastrointestinal tract (disorder) → Disorder of abdominopelvic segment of trunk (disorder) → Disorder of trunk (disorder) → Finding of trunk structure (finding)\n2) 432414001 | Concept name: Constipation due to atony of colon (disorder) | Score: 1.013 | Ancestor path: Constipation due to atony of colon (disorder) → Functional disorder of intestine (disorder) → Disorder of intestine (disorder) → Disorder of abdomen (disorder) → Finding of abdomen (

In [48]:
AUDIT_1 = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
AUDIT_2 = "results/20260415_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "e3de56de-b707-405e-bf97-837070e5c744"
item_index = 5
call_name = "direct_match"
print(f"Are system prompts identical? {get_values(AUDIT_1, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt') == get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name, keys=['system_prompt']).get('system_prompt')}")
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Simple Dense Index:", vals)
print("\n------------------------------------------")
vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("Enhanced Dense Index:", vals)
# print("\n------------------------------------------")
# call_name = "route_or_fill"
# vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
#                   keys=["user_prompt","parsed_output"])
# print("values:", vals)

Are system prompts identical? True
Simple Dense Index: {'user_prompt': 'QUERY:\n"decompensated heart failure"\n\nCANDIDATES (choose from these only):\n1) 424404003 | Concept name: Decompensated chronic heart failure (disorder) | Score: 8.088 | Ancestor path: Decompensated chronic heart failure (disorder) → Chronic heart failure (disorder) → Chronic heart disease (disorder) → Heart disease (disorder) → Disorder of mediastinum (disorder) → Disorder of thorax (disorder) → Finding of thoracic region (finding) → Finding of upper trunk (finding) → Finding of trunk structure (finding)\n2) 195111005 | Concept name: Decompensated cardiac failure (disorder) | Score: 7.989 | Ancestor path: Decompensated cardiac failure (disorder) → Heart failure (disorder) → Disorder of cardiac function (disorder) → Heart disease (disorder) → Disorder of mediastinum (disorder) → Disorder of thorax (disorder) → Finding of thoracic region (finding) → Finding of upper trunk (finding) → Finding of trunk structure (fi

In [ ]:
AUDIT = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "6ae413ba-5a48-423f-8b8d-7a42661cf836"
item_index = 0
call_name = "direct_match"
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt", "user_prompt","parsed_output"])
print("values:", vals)
# print("\n------------------------------------------")
# call_name = "route_or_fill"
# vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
#                   keys=["user_prompt","parsed_output"])
# print("values:", vals)

values: {'system_prompt': '\nYou are a strict biomedical terminology validator.\nYour role is to determine if a candidate is a LEXICAL and SEMANTIC identity match for the contraindication query.\n\nYour ONLY task:\n- Identify if a DIRECT MATCH exists among the provided candidates.\n- If a match exists, select exactly ONE candidate from the list.\n- Otherwise, return no match.\n\n--------------------\nDIRECT MATCH (STRICT)\n--------------------\nA candidate is a DIRECT MATCH only if it is an exact semantic equivalent. Do NOT bridge concepts even if they are clinically related.\n\n1) Lexical-Semantic Alignment:\n- The candidate must match the specificity and naming of the query.\n- Do NOT bridge concepts that use different primary terms even if they are clinically related.\n- If the query text and candidate label belong to different levels of the hierarchy, return no match.\n2) Temporal/Contextual Scope:\n- "Post-X" is NOT a match for "X".\n- "History of X" is NOT a match for "X".\n- If 

In [100]:
AUDIT = "results/20260414_isolation/100_bm25_only/runtime_audit.jsonl"
SET_ID = "fe08bcd5-4d14-4c2d-940c-b7968f9a89f0"
item_index = 4
call_name = "route_or_fill"
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)
# print("\n------------------------------------------")
# call_name = "route_or_fill"
# vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
#                   keys=["user_prompt","parsed_output"])
# print("values:", vals)

values: {'user_prompt': 'QUERY:\nMycobacterial infection of the eye\n\nEXTRACTED_FIELDS:\ncontraindication_state_text=infection\nsubstance_text=Mycobacterial\nseverity_span=None\ncourse_span=None\n\nATTRIBUTE_TABLE:\n{"causative_agent":246075003,"severity":246112005,"clinical_course":263502005}\n\nFOCUS_CANDIDATES:\n1) 838338001 | Concept name: Bacterial infection due to human immunodeficiency virus infection (disorder) | Score: 2.612\n2) 63459005 | Concept name: Clinical infection (disorder) | Score: 1.891\n3) 275393007 | Concept name: Oral infection (disorder) | Score: 1.444\n4) 312155003 | Concept name: Genital infection (disorder) | Score: 0.974\n5) 10443009 | Concept name: Localized infection (disorder) | Score: 0.844\n6) 76844004 | Concept name: Local infection of wound (disorder) | Score: 0.423\n7) 260976002 | Concept name: Presence of infection (qualifier value) | Score: 0.027\n8) 186747009 | Concept name: Coronavirus infection (disorder) | Score: -0.089\n9) 41659003 | Concept 

In [62]:
# strict barely better than anc onnly
AUDIT = "results/20260414_isolation/001_strict_only/runtime_audit.jsonl"
AUDIT_2 = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "fe08bcd5-4d14-4c2d-940c-b7968f9a89f0"
item_index = 4
call_name = "direct_match"
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt"])
print("values:", vals)
print("\n------------------------------------------")
# call_name = "route_or_fill"
vals = get_values(AUDIT_2, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["system_prompt"])
print("values:", vals)

values: {'system_prompt': '\nYou are a strict biomedical terminology validator.\nYour role is to determine if a candidate is a LEXICAL and SEMANTIC identity match for the contraindication query.\n\nYour ONLY task:\n- Identify if a DIRECT MATCH exists among the provided candidates.\n- If a match exists, select exactly ONE candidate from the list.\n- Otherwise, return no match.\n\nWhen in doubt, return no match. A false negative here is recoverable; a false positive is not.\n\n--------------------\nDIRECT MATCH (STRICT)\n--------------------\nA candidate is a DIRECT MATCH only if its label is an exact lexical-semantic equivalent to the query — identical clinical meaning, identical ontological level, identical primary term.\n\n1) Primary Term Identity:\n- The root condition word used in the query MUST be the same root condition word in the candidate label.\n- Clinically related terms that are NOT interchangeable in SNOMED CT:\n    * "Hypersensitivity" ≠ "Allergy"  (Allergy is a subtype of

#### Hypersensitivity not getting pulled


In [36]:
# No hypersensitivity 
AUDIT = "results/20260414_isolation/100_bm25_only/runtime_audit.jsonl"
# AUDIT_2 = "results/20260414_isolation/010_anc_only/runtime_audit.jsonl"
SET_ID = "b0c91667-ab6f-481d-a592-62da8a52abf7"
item_index = 1
call_name = "direct_match"
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)
print("\n------------------------------------------")
call_name = "route_or_fill"
vals = get_values(AUDIT, SET_ID, item_index=item_index, call_name=call_name,
                  keys=["user_prompt","parsed_output"])
print("values:", vals)

values: {'user_prompt': 'QUERY:\n"Known hypersensitivity to atropine"\n\nCANDIDATES (choose from these only):\n1) 1269398002 | Concept name: Hypersensitivity to minocycline (finding) | Score: -0.409\n2) 1269354002 | Concept name: Hypersensitivity to erythromycin (finding) | Score: -1.791\n3) 1269353008 | Concept name: Hypersensitivity to gentamicin (finding) | Score: -2.071\n4) 1269352003 | Concept name: Hypersensitivity to succinylcholine (finding) | Score: -2.394\n5) 114931000119103 | Concept name: Hypersensitivity to abacavir (finding) | Score: -2.559\n6) 1269395004 | Concept name: Hypersensitivity to hydrocortisone (finding) | Score: -2.713\n7) 292534001 | Concept name: Atropine adverse reaction (disorder) | Score: -2.784\n8) 1269400003 | Concept name: Hypersensitivity to nalidixic acid (finding) | Score: -2.878\n9) 609548005 | Concept name: Non-allergic hypersensitivity to aspartame (finding) | Score: -2.981\n10) 1163445008 | Concept name: Hypersensitivity to polysorbate-80 (findi

In [3]:
# onetime fix for agg_csv 
import subprocess
def run_rewrite(input_file, output_file):
    cmd = [
        "python3",
        "rewrite_agg_csv.py",
        "--in", input_file,
        "--out", output_file
    ]
    subprocess.run(cmd, check=True)

In [11]:
# get list of all jsonl files from a given directory. Have subdirectories too, so need to walk through them
# Need to specify a directory to look into
import os
target = "results/20260414_isolation"
for root, dirs, files in os.walk(target):
    for filename in files:
        if filename == "aggregated_results.jsonl":
            input = os.path.join(root, filename)
            output = os.path.join(root, "aggregated_results_rewrite.csv")
            run_rewrite(input, output)
            print(f"Found file: {input}")
            print(f"Rewriting to: {output}")

Read 98 SPL groups (385 items) from: results/20260414_isolation/101_bm25_strict/aggregated_results.jsonl
Wrote CSV (385 rows) to: results/20260414_isolation/101_bm25_strict/aggregated_results_rewrite.csv
Found file: results/20260414_isolation/101_bm25_strict/aggregated_results.jsonl
Rewriting to: results/20260414_isolation/101_bm25_strict/aggregated_results_rewrite.csv
Read 98 SPL groups (385 items) from: results/20260414_isolation/110_bm25_anc/aggregated_results.jsonl
Wrote CSV (385 rows) to: results/20260414_isolation/110_bm25_anc/aggregated_results_rewrite.csv
Found file: results/20260414_isolation/110_bm25_anc/aggregated_results.jsonl
Rewriting to: results/20260414_isolation/110_bm25_anc/aggregated_results_rewrite.csv
Read 98 SPL groups (385 items) from: results/20260414_isolation/111_all_on/aggregated_results.jsonl
Wrote CSV (385 rows) to: results/20260414_isolation/111_all_on/aggregated_results_rewrite.csv
Found file: results/20260414_isolation/111_all_on/aggregated_results.jsonl

In [50]:
from agent_runner import evaluate_aggregated_predictions
import json
import os
# gold_csv = "results/contra_gold_100_2.csv"
gold_csv = "results/contra_gold_100_3.csv"
# aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
# eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
# eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"
targets = ["results/20260414_isolation", "results/20260415_isolation"]
bools = [True, False]
for target in targets:
    for decoupled in bools:
        for root, dirs, files in os.walk(target):
            for filename in files:
                if filename == "aggregated_results_rewrite.csv":
                    aggregated_csv = os.path.join(root, filename)
                    json_name = "evaluation_metrics_decoupled.json" if decoupled else "evaluation_metrics_coupled.json"
                    csv_name = "evaluation_details_decoupled.csv" if decoupled else "evaluation_details_coupled.csv"
                    eval_json = os.path.join(root, json_name)
                    eval_details_csv = os.path.join(root, csv_name)
                    metrics = evaluate_aggregated_predictions(
                                pred_csv=aggregated_csv,
                                gold_csv=gold_csv,
                                out_json=eval_json,
                                out_details_csv=eval_details_csv,
                                decoupled=decoupled,
                                min_pair_score=0.8
                            )
                    print(json.dumps(metrics, indent=2))
                    print(f"Wrote evaluation metrics to: {eval_json}")
                    print(f"Wrote evaluation details to: {eval_details_csv}")

Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7587.56it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260414_isolation/101_bm25_strict/aggregated_results_rewrite.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": true,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_l

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4900.31it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260414_isolation/110_bm25_anc/aggregated_results_rewrite.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": true,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_leve

...

Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260414_isolation/001_strict_only/aggregated_results_rewrite.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": false,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 8057.58it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260414_isolation/010_anc_only/aggregated_results_rewrite.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": false,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_lev

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7964.17it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260414_isolation/100_bm25_only/aggregated_results_rewrite.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": false,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_le

In [57]:
from agent_runner import evaluate_aggregated_predictions
import json
import os
# gold_csv = "results/contra_gold_100_2.csv"
gold_csv = "results/contra_gold_100_3.csv"
# aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
# eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
# eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"
targets = ["results/20260415_isolation"]
bools = [True, False]
for target in targets:
    for decoupled in bools:
        for root, dirs, files in os.walk(target):
            for filename in files:
                if filename == "aggregated_hits.csv":
                    aggregated_csv = os.path.join(root, filename)
                    json_name = "evaluation_metrics_decoupled.json" if decoupled else "evaluation_metrics_coupled.json"
                    csv_name = "evaluation_details_decoupled.csv" if decoupled else "evaluation_details_coupled.csv"
                    eval_json = os.path.join(root, json_name)
                    eval_details_csv = os.path.join(root, csv_name)
                    metrics = evaluate_aggregated_predictions(
                                pred_csv=aggregated_csv,
                                gold_csv=gold_csv,
                                out_json=eval_json,
                                out_details_csv=eval_details_csv,
                                decoupled=decoupled,
                                min_pair_score=0.8
                            )
                    print(json.dumps(metrics, indent=2))
                    print(f"Wrote evaluation metrics to: {eval_json}")
                    print(f"Wrote evaluation details to: {eval_details_csv}")

Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 9192.83it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260415_isolation/101_bm25_strict/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": true,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_level": {
  

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7665.47it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260415_isolation/110_bm25_anc/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": true,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_level": {
    "

...

Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260415_isolation/111_all_on/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_3.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": true,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 232,
    "fp": 117,
    "fn": 82,
    "precision": 0.664756446991404,
    "recall": 0.7388535031847133,
    "f1": 0.6998491704374058
  },
  "contraindication_level": {
    "tp

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 5693.46it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).


Traceback (most recent call last):
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 858, in <module>
    main()
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 657, in main
    compute_tiered_concept_metrics(
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 325, in compute_tiered_concept_metrics
    [
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 326, in <listcomp>
    [
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 327, in <listcomp>
    concept_similarity_score(
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 296, in concept_similarity_score
    lca = max(common, key=lambda a: get_concept_depth(a, rel_df_indexed))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/wmanuel3/VaxMapperRepo/evaluate_agg_results_3.py", line 296, in <lambda>
    lca = max(common, key=lambda a: get_concept_depth(a, rel_df_indexed))
       

KeyboardInterrupt: 

In [5]:
import json
import os
import pandas as pd

target = "results/20260414_isolation"
rows = []

for root, dirs, files in os.walk(target):
    for filename in files:
        if filename == "evaluation_metrics_2.json":
            method = os.path.basename(root)
            eval_json = os.path.join(root, filename)

            with open(eval_json) as f:
                metrics = json.load(f)

            rows.append({
                ("method", ""): method,

                ("extraction", "precision"): metrics.get("extraction_level", {}).get("precision"),
                ("extraction", "recall"): metrics.get("extraction_level", {}).get("recall"),
                ("extraction", "f1"): metrics.get("extraction_level", {}).get("f1"),

                ("concept", "precision"): metrics.get("concept_level", {}).get("precision"),
                ("concept", "recall"): metrics.get("concept_level", {}).get("recall"),
                ("concept", "f1"): metrics.get("concept_level", {}).get("f1"),

                ("contraindication", "precision"): metrics.get("contraindication_level", {}).get("precision"),
                ("contraindication", "recall"): metrics.get("contraindication_level", {}).get("recall"),
                ("contraindication", "f1"): metrics.get("contraindication_level", {}).get("f1"),
            })

df = pd.DataFrame(rows)
df = df.sort_values(("method", "")).reset_index(drop=True)
df


,"(method, )","(extraction, precision)","(extraction, recall)","(extraction, f1)","(concept, precision)","(concept, recall)","(concept, f1)","(contraindication, precision)","(contraindication, recall)","(contraindication, f1)"
0,000_baseline,0.664756,0.738854,0.699849,0.834234,0.734921,0.781435,0.551724,0.551724,0.551724
1,001_strict_only,0.664756,0.738854,0.699849,0.849713,0.782275,0.814601,0.573276,0.573276,0.573276
2,010_anc_only,0.664756,0.738854,0.699849,0.846142,0.754365,0.797622,0.556034,0.556034,0.556034
3,011_anc_strict,0.662824,0.732484,0.695915,0.862500,0.793316,0.826462,0.578261,0.578261,0.578261
4,100_bm25_only,0.664756,0.738854,0.699849,0.831325,0.730159,0.777465,0.560345,0.560345,0.560345
5,101_bm25_strict,0.664756,0.738854,0.699849,0.835942,0.762963,0.797787,0.564655,0.564655,0.564655
6,110_bm25_anc,0.664756,0.738854,0.699849,0.827126,0.746164,0.784562,0.560345,0.560345,0.560345
7,111_all_on,0.664756,0.738854,0.699849,0.834091,0.776720,0.804384,0.568966,0.568966,0.568966


In [56]:
df

,"(method, )","(extraction, precision)","(extraction, recall)","(extraction, f1)","(concept, precision)","(concept, recall)","(concept, f1)","(contraindication, precision)","(contraindication, recall)","(contraindication, f1)"
0,000_baseline,0.664756,0.738854,0.699849,0.792104,0.697831,0.741985,0.551724,0.551724,0.551724
1,001_strict_only,0.664756,0.738854,0.699849,0.818156,0.753240,0.784357,0.573276,0.573276,0.573276
2,010_anc_only,0.664756,0.738854,0.699849,0.805160,0.717874,0.759016,0.556034,0.556034,0.556034
3,011_anc_strict,0.662824,0.732484,0.695915,0.829895,0.763362,0.795239,0.578261,0.578261,0.578261
4,100_bm25_only,0.664756,0.738854,0.699849,0.786510,0.690823,0.735567,0.560345,0.560345,0.560345
5,101_bm25_strict,0.664756,0.738854,0.699849,0.798754,0.729040,0.762306,0.564655,0.564655,0.564655
6,110_bm25_anc,0.664756,0.738854,0.699849,0.782825,0.706240,0.742563,0.560345,0.560345,0.560345
7,111_all_on,0.664756,0.738854,0.699849,0.795713,0.741010,0.767388,0.568966,0.568966,0.568966


In [22]:
df.to_csv("results/20260414_isolation/summary_results_2.csv", index=False)

In [23]:
from evaluate_agg_results_3 import compute_tiered_concept_metrics

In [ ]:
compute_tiered_concept_metrics([])

In [ ]:
# presentation evaluation: 

In [ ]:
from agent_runner import evaluate_aggregated_predictions
import json
import os
# gold_csv = "results/contra_gold_100_2.csv"
# medgemma baseline
gold_csv = "results/contra_gold_100_2.csv"
# aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
# eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
# eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"
targets = ["results/20260402"]
bools = [False]
for target in targets:
    for decoupled in bools:
        for root, dirs, files in os.walk(target):
            for filename in files:
                if filename == "aggregated_hits.csv":
                    aggregated_csv = os.path.join(root, filename)
                    json_name = "evaluation_metrics_decoupled.json" if decoupled else "evaluation_metrics_coupled.json"
                    csv_name = "evaluation_details_decoupled.csv" if decoupled else "evaluation_details_coupled.csv"
                    eval_json = os.path.join(root, json_name)
                    eval_details_csv = os.path.join(root, csv_name)
                    metrics = evaluate_aggregated_predictions(
                                pred_csv=aggregated_csv,
                                gold_csv=gold_csv,
                                out_json=eval_json,
                                out_details_csv=eval_details_csv,
                                decoupled=decoupled,
                                min_pair_score=0.8
                            )
                    print(json.dumps(metrics, indent=2))
                    print(f"Wrote evaluation metrics to: {eval_json}")
                    print(f"Wrote evaluation details to: {eval_details_csv}")

/home/srv-wmanuel3/miniconda3/envs/splmap/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 8588.10it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260402/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_2.csv",
    "gold_rows_total": 405,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 91,
    "discard_na_gold_expression": true,
    "pred_rows": 385,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": false,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 177,
    "fp": 183,
    "fn": 137,
    "precision": 0.49166666666666664,
    "recall": 0.5636942675159236,
    "f1": 0.5252225519287833
  },
  "contraindication_level": {
    "tp": 86,
    "fp": 

In [ ]:
from agent_runner import evaluate_aggregated_predictions
import json
import os
# gold_csv = "results/contra_gold_100_2.csv"
# gemma4 baseline
gold_csv = "results/contra_gold_100_2.csv"
# aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
# eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
# eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"
targets = ["results/20260406"]
bools = [False]
for target in targets:
    for decoupled in bools:
        for root, dirs, files in os.walk(target):
            for filename in files:
                if filename == "aggregated_hits.csv":
                    aggregated_csv = os.path.join(root, filename)
                    json_name = "evaluation_metrics_decoupled.json" if decoupled else "evaluation_metrics_coupled.json"
                    csv_name = "evaluation_details_decoupled.csv" if decoupled else "evaluation_details_coupled.csv"
                    eval_json = os.path.join(root, json_name)
                    eval_details_csv = os.path.join(root, csv_name)
                    metrics = evaluate_aggregated_predictions(
                                pred_csv=aggregated_csv,
                                gold_csv=gold_csv,
                                out_json=eval_json,
                                out_details_csv=eval_details_csv,
                                decoupled=decoupled,
                                min_pair_score=0.8
                            )
                    print(json.dumps(metrics, indent=2))
                    print(f"Wrote evaluation metrics to: {eval_json}")
                    print(f"Wrote evaluation details to: {eval_details_csv}")

Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7885.78it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260406/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_2.csv",
    "gold_rows_total": 405,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 91,
    "discard_na_gold_expression": true,
    "pred_rows": 391,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": false,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 239,
    "fp": 116,
    "fn": 75,
    "precision": 0.6732394366197183,
    "recall": 0.7611464968152867,
    "f1": 0.7144992526158445
  },
  "contraindication_level": {
    "tp": 121,
    "fp": 2

In [ ]:
from agent_runner import evaluate_aggregated_predictions
import json
import os
# gold_csv = "results/contra_gold_100_2.csv"
# qwen 3.5
gold_csv = "results/contra_gold_100_2.csv"
# aggregated_csv = "results/20260407/gemma4_bm25_ancestor_paths/aggregated_hits.csv"
# eval_json = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_metrics_gemma4dcf.json"
# eval_details_csv = "results/20260407/gemma4_bm25_ancestor_paths/evaluation_details_gemma4dcf.csv"
targets = ["results/20260401"]
bools = [False]
for target in targets:
    for decoupled in bools:
        for root, dirs, files in os.walk(target):
            for filename in files:
                if filename == "aggregated_hits.csv":
                    aggregated_csv = os.path.join(root, filename)
                    json_name = "evaluation_metrics_decoupled.json" if decoupled else "evaluation_metrics_coupled.json"
                    csv_name = "evaluation_details_decoupled.csv" if decoupled else "evaluation_details_coupled.csv"
                    eval_json = os.path.join(root, json_name)
                    eval_details_csv = os.path.join(root, csv_name)
                    metrics = evaluate_aggregated_predictions(
                                pred_csv=aggregated_csv,
                                gold_csv=gold_csv,
                                out_json=eval_json,
                                out_details_csv=eval_details_csv,
                                decoupled=decoupled,
                                min_pair_score=0.8
                            )
                    print(json.dumps(metrics, indent=2))
                    print(f"Wrote evaluation metrics to: {eval_json}")
                    print(f"Wrote evaluation details to: {eval_details_csv}")

Loading model google/embeddinggemma-300m...


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 10725.46it/s]


Model google/embeddinggemma-300m loaded to cuda:0 successfully.
Tiered concept scoring enabled (631,617 IS-A rows loaded).
{
  "inputs": {
    "pred_csv": "results/20260401/aggregated_hits.csv",
    "gold_csv": "results/contra_gold_100_2.csv",
    "gold_rows_total": 405,
    "gold_rows_used": 314,
    "gold_rows_dropped_na_expression": 91,
    "discard_na_gold_expression": true,
    "pred_rows": 392,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "st_model_id": "google/embeddinggemma-300m",
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.8,
    "decoupled": false,
    "assignment_method": "hungarian",
    "tiered_concept_scoring": true,
    "hierarchy_partial_score": 0.0
  },
  "extraction_level": {
    "tp": 231,
    "fp": 129,
    "fn": 83,
    "precision": 0.6416666666666667,
    "recall": 0.7356687898089171,
    "f1": 0.685459940652819
  },
  "contraindication_level": {
    "tp": 122,
    "fp": 23